# Lesson 15 — Presolve and FBBT

## Learning objectives

1. Recognize the standard presolve transformations.
2. Implement feasibility-based bound tightening (FBBT) by hand.
3. Understand probing and its trade-offs.
4. Use `discopt`'s presolve flags to debug bad models.

## 1. Why presolve matters

Presolve runs *before* B&B and aims to reduce problem size, tighten bounds, and detect infeasibility. On real MILPs, presolve commonly reduces variable count 30–60% and constraint count 20–50% {cite:p}`Savelsbergh1994,Achterberg2007`. Time spent in presolve is almost always recovered.

## 2. Standard transformations

| Transformation | Example |
|----------------|---------|
| Singleton row | $x_j \ge 5$ from a single-variable constraint → tighten bound |
| Empty column | Variable with 0 coefficients → fix to bound that helps objective |
| Implied bound | $x_1 + x_2 + x_3 = 1, x_i \in [0,1]$ → already tight |
| Coefficient strengthening | Loose big-M → tighten via implied bounds |
| Aggregation | $x_1 = 2 x_2 + 3$ → eliminate $x_1$ |
| Probing | Set $x_j = 0$, propagate; if infeasible, fix $x_j = 1$ |

See {cite:p}`Savelsbergh1994` and {cite:p}`Belotti2009` for the systematic catalogue.

## 3. Feasibility-based bound tightening (FBBT)

Given a constraint $\sum_j a_j x_j \le b$ with $x_j \in [\ell_j, u_j]$, the implied bound on each $x_k$ is

$$x_k \le \frac{b - \sum_{j \ne k, a_j > 0} a_j \ell_j - \sum_{j \ne k, a_j < 0} a_j u_j}{a_k} \quad \text{(if } a_k > 0\text{)}.$$

(Mirror for $a_k < 0$ and for $\ge$ constraints.) Apply to every variable in every constraint; iterate to a fixed point. {cite:p}`Belotti2009` formalize this for the MINLP setting.

## 4. Probing

Pick a binary $x_j$. Tentatively set $x_j = 0$, run FBBT. If infeasible, fix $x_j = 1$. Repeat for $x_j = 1$. Either reduces variables or generates *implied bounds* and *cliques* ($x_j = 1 \Rightarrow x_k = 0$). Quadratic in the number of binaries; usually limited to a budget {cite:p}`Achterberg2007`.

### Presolve in `discopt`

Presolve runs *inside* `m.solve()`. The orchestrator in the Rust backend (`crates/discopt-core/src/presolve/orchestrator.rs`) drives a list of named passes to a *fixed point* under a global iteration, time and work budget. The passes include variable elimination, factorable elimination, simplification, coefficient strengthening, FBBT, probing, aggregation, implied bounds, clique detection, redundancy removal, scaling and symmetry detection. The pass catalogue lives under `crates/discopt-core/src/presolve/`; `ROADMAP.md` there tracks status.

The Python-visible knobs on `m.solve()` are:

| Argument | Default | What it does |
|---|---|---|
| `presolve=True` | on | master switch for `discopt`'s root-presolve pipeline |
| `presolve_polynomial=False` | off | enables the (slower) polynomial / factorable reformulation passes |
| `presolve_reverse_ad=False` | off | reverse-AD-based bound tightening on top of FBBT |
| `in_tree_presolve_stride=1` | every node | FBBT-only presolve every `k` B&B nodes (`0` = root only) |
| `highs_options={"presolve": "off"}` | HiGHS presolves | switches off *HiGHS's* presolve on the HiGHS LP/MILP route |

**Which route are you on?** A *pure* LP or MILP goes to HiGHS by default, and HiGHS runs its own presolve. `discopt` therefore skips its root presolve there, apart from big-M coefficient tightening, so on that route `presolve=False` changes essentially nothing. To switch off the presolve that actually runs there, HiGHS's own, pass `highs_options={"presolve": "off"}`. With `milp_backend="native"`, `discopt`'s own MILP tree runs instead, and there `presolve=False` does switch off `discopt`'s root presolve. The flag also matters on the nonlinear / spatial branch-and-bound routes. Even there, expect small effects on small models: FBBT and OBBT run again at the root and FBBT runs in every node, so a skipped root pass is often recovered later.

**Inspecting what presolve does.** Two public helpers in `discopt.tightening` run the bound-tightening core on a model without solving it:

- `fbbt_box(m)` runs FBBT to a fixpoint and returns the tightened flat bounds `lb`, `ub`, an `infeasible` flag and `n_tightened`;
- `probe_box(m)` runs FBBT plus probing on the discrete variables (§4).

Both are meant to be *sound*: they should never remove a feasible point. Both work per element, on array variables as well as scalars. The demo after §5 uses `fbbt_box`. To watch the whole pass schedule, `discopt._rust.model_to_repr(m).presolve()` returns the presolved representation and a per-pass delta log (see the `presolve` notebook in the `discopt` docs).

**When a model is wrong, not just slow:** `fbbt_box(m).infeasible` is a cheap first check, since FBBT alone often proves infeasibility. `discopt.infeasibility.compute_iis(m)` finds an irreducible infeasible subset of constraints and bounds. `m.solve(debug=True)` opens the interactive branch-and-bound debugger (`discopt.debug`), which pauses at node checkpoints so you can inspect bounds, relaxations and the incumbent.

## 5. FBBT in MINLP

For nonlinear constraints, FBBT uses **interval arithmetic** {cite:p}`Moore1966,Neumaier1990`. Given $z = x \cdot y$ with $x \in [a, b], y \in [c, d]$, the implied bound is $z \in [\min(ac, ad, bc, bd), \max(...)]$.

Interval arithmetic suffers from the **dependency problem**. It treats every occurrence of a variable as independent, so $x \cdot x$ with $x \in [-10, 10]$ evaluates to $[-100, 100]$, while $x^2$ gives the exact $[0, 100]$. How you *write* a term changes how much FBBT can do (see the demo below). Enclosures can also be loose for high-rank monomials. **Optimization-based BT (OBBT)** instead solves a small auxiliary LP/NLP to compute each bound — much tighter, much slower.

In [ ]:
# FBBT with discopt.tightening: the same constraint written two ways.
import discopt as do
from discopt.tightening import fbbt_box

for label, square in [("x**2 <= y", lambda x: x**2), ("x*x  <= y", lambda x: x * x)]:
    m = do.Model("fbbt_demo")
    x = m.continuous("x", lb=-10, ub=10)
    y = m.continuous("y", lb=0.5, ub=4)
    m.subject_to(square(x) <= y)
    m.minimize(x)
    bt = fbbt_box(m)
    print(f"{label}:  x in [{bt.lb[0]:g}, {bt.ub[0]:g}]   (n_tightened={bt.n_tightened})")

## 6. Effects on B&B

Tighter bounds → tighter LP relaxation → more pruning → smaller tree. Plus tighter bounds → tighter big-M and McCormick relaxations → tighter convex envelopes.

This is why "pre-modelling" (giving good initial bounds in your formulation) is usually the highest-leverage thing you can do for solver performance.

## References
{cite:p}`Savelsbergh1994` (LP/MILP), {cite:p}`Belotti2009` (MINLP), {cite:p}`Achterberg2007`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm
